# ⚡ Automated Mamba Build & Pack for Google Colab (T4 GPU / PyTorch 2.14+ / CUDA 13.0)

Notebook này dùng để **biên dịch Mamba (`mamba-ssm` + `causal-conv1d`) CHUẨN CUDA kernel một lần duy nhất** trên môi trường mới của Google Colab.

### 💡 Tại sao cần notebook này?
- Google Colab đã tự động nâng cấp môi trường lên **Python 3.13 + PyTorch 2.14.x + CUDA 13.0**.
- File zip cũ (`torch 2.11`) bị lỗi ABI C++ (`undefined symbol: _ZN3c104cuda...`).
- Quá trình compile thông thường mất **2 - 3 tiếng** vì mặc định PyTorch compile cho mọi kiến trúc GPU.
- **Script này đã được tối ưu hóa:** Sử dụng `TORCH_CUDA_ARCH_LIST="7.5"` để **chỉ compile duy nhất cho GPU Tesla T4**, rút ngắn thời gian xuống chỉ còn **~15 - 20 phút**!
- Sau khi compile xong, script sẽ test GPU thật, tự động đóng gói đúng chuẩn `.zip` và tự động kích hoạt tải về máy hoặc lưu vào Google Drive của bạn.
- Sau này, bạn chỉ cần nạp file zip này vào các notebook nghiên cứu trong **1 - 2 phút** mà không cần compile lại bao giờ nữa!

---

In [ ]:
# ==============================================================================
# 🚀 CELL TỰ ĐỘNG BIÊN DỊCH, TEST VÀ ĐÓNG GÓI MAMBA T4 CHO COLAB MỚI
# ==============================================================================
import os, sys, sysconfig, pathlib, subprocess, json, shutil, zipfile, time, importlib.metadata
import torch

print("=== BƯỚC 1: KIỂM TRA PHẦN CỨNG GPU T4 ===")
assert torch.cuda.is_available(), "❌ Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> Chọn T4 GPU."
gpu_cap = torch.cuda.get_device_capability(0)
gpu_name = torch.cuda.get_device_name(0)
print(f"✅ GPU phát hiện: {gpu_name} (Compute Capability: {gpu_cap})")
assert gpu_cap == (7, 5), f"❌ Cảnh báo: GPU hiện tại là {gpu_name} ({gpu_cap}), không phải T4 (7, 5). Hãy đổi sang T4!"

print("\n=== BƯỚC 2: CÀI ĐẶT BUILD TOOLS & THIẾT LẬP TỐI ƯU TỐC ĐỘ BIÊN DỊCH ===")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ninja", "packaging", "wheel", "setuptools", "einops", "transformers", "triton"], check=True)

# ⚡ TỐI ƯU CỰC KỲ QUAN TRỌNG: Giới hạn chỉ compile cho T4 (sm_75)
# Giúp rút ngắn thời gian từ 2-3 tiếng xuống chỉ còn ~15 - 20 phút!
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"
os.environ["MAX_JOBS"] = str(os.cpu_count() or 2)
print(f"✅ Đã thiết lập TORCH_CUDA_ARCH_LIST={os.environ['TORCH_CUDA_ARCH_LIST']}")
print(f"✅ Đã thiết lập MAX_JOBS={os.environ['MAX_JOBS']}")

print("\n=== BƯỚC 3: BIÊN DỊCH CAUSAL-CONV1D (CHUẨN CUDA) ===")
t0 = time.time()
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-build-isolation", "--no-cache-dir", "causal-conv1d>=1.4.0"
], check=True)
print(f"✅ causal-conv1d hoàn tất trong {int(time.time() - t0)} giây.")

print("\n=== BƯỚC 4: BIÊN DỊCH MAMBA-SSM (CHUẨN CUDA) ===")
t1 = time.time()
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "--no-build-isolation", "--no-cache-dir", "mamba-ssm>=2.2.0"
], check=True)
print(f"✅ mamba-ssm hoàn tất trong {int(time.time() - t1)} giây.")

print("\n=== BƯỚC 5: KIỂM TRA THỰC TẾ KERNEL CUDA TRÊN GPU ===")
import selective_scan_cuda, causal_conv1d_cuda
from mamba_ssm.modules.mamba_simple import causal_conv1d_fn
from mamba_ssm import Mamba

print("causal_conv1d_fn:", "OK (CUDA kernel)" if causal_conv1d_fn is not None else "❌ LỖI: FALLBACK!")
assert causal_conv1d_fn is not None, "causal_conv1d_fn bị None, chưa nạp được kernel CUDA!"

test_x = torch.randn(2, 128, 64, device="cuda")
test_model = Mamba(d_model=64).cuda()
test_out = test_model(test_x)
test_out.mean().backward()
print("✅ MAMBA FORWARD & BACKWARD HOẠT ĐỘNG HOÀN HẢO! Output shape:", test_out.shape)

print("\n=== BƯỚC 6: ĐÓNG GÓI PACK .ZIP VÀ TẠO MANIFEST ===")
SP = pathlib.Path(sysconfig.get_paths()["purelib"])
print(f"Site-packages đường dẫn: {SP}")

try:
    mamba_v = importlib.metadata.version("mamba-ssm")
except Exception:
    mamba_v = "2.3.2"
try:
    conv_v = importlib.metadata.version("causal-conv1d")
except Exception:
    conv_v = "1.7.0"

manifest = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "dists": {
        "mamba-ssm": mamba_v,
        "causal-conv1d": conv_v
    }
}
print("Nội dung manifest:")
print(json.dumps(manifest, indent=2))

py_str = f"py{sys.version_info.major}{sys.version_info.minor}"
torch_str = f"torch{torch.__version__.split('+')[0]}"
cuda_str = f"cu{torch.version.cuda.replace('.', '')}" if torch.version.cuda else "cu130"
zip_filename = f"mamba_{py_str}_{torch_str}_{cuda_str}_T4.zip"
zip_dest = pathlib.Path(f"/content/{zip_filename}")

items_to_pack = []
for pkg_name in ["mamba_ssm", "causal_conv1d"]:
    d = SP / pkg_name
    if d.exists() and d.is_dir():
        items_to_pack.append(d)

for dinfo in SP.glob("mamba_ssm*.dist-info"):
    items_to_pack.append(dinfo)
for dinfo in SP.glob("causal_conv1d*.dist-info"):
    items_to_pack.append(dinfo)

for so in SP.glob("selective_scan_cuda*.so"):
    items_to_pack.append(so)
for so in SP.glob("causal_conv1d_cuda*.so"):
    items_to_pack.append(so)

print(f"\nTìm thấy {len(items_to_pack)} mục chính để đóng gói:")
for item in items_to_pack:
    print(f" - {item.name}")

print(f"\n>>> Đang nén file {zip_dest}...")
with zipfile.ZipFile(zip_dest, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.writestr("__manifest__.json", json.dumps(manifest, indent=2))
    for item in items_to_pack:
        if item.is_file():
            z.write(item, arcname=item.name)
        elif item.is_dir():
            for root, dirs, files in os.walk(item):
                for f in files:
                    full_file = pathlib.Path(root) / f
                    rel = full_file.relative_to(SP)
                    z.write(full_file, arcname=str(rel).replace("\\", "/"))

size_mb = zip_dest.stat().st_size / (1024 * 1024)
print(f"🎉 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {zip_dest} ({size_mb:.1f} MB)")

print("\n=== BƯỚC 7: TẢI VỀ MÁY TÍNH / LƯU GOOGLE DRIVE ===")
# Cách 1: Lưu thẳng vào Google Drive (tiện nhất để lấy FILE_ID)
try:
    from google.colab import drive
    print("Mount Google Drive...")
    drive.mount('/content/drive')
    drive_path = f"/content/drive/MyDrive/{zip_filename}"
    shutil.copy(zip_dest, drive_path)
    print(f"✅ ĐÃ COPY THÀNH CÔNG VÀO GOOGLE DRIVE: {drive_path}")
except Exception as e:
    print("Bỏ qua mount Google Drive:", e)

# Cách 2: Tự động kích hoạt tải về máy qua browser
try:
    from google.colab import files
    print(f"Đang tải file {zip_filename} về máy tính của bạn...")
    files.download(str(zip_dest))
except Exception as e:
    print("Lưu ý khi tải browser:", e)


### 📦 CELL KIỂM TRA NHANH: NẠP LẠI PACK ĐÃ TẠO (CHỈ MẤT 1 PHÚT)
> Dùng cell dưới đây trong bất kỳ notebook nào để nạp Mamba trong 1 - 2 phút (chỉ cần thay `FILE_ID` của bạn trên Google Drive):

In [ ]:
# ==============================================================================
# CELL CÀI ĐẶT MAMBA NHANH TRONG 1 PHÚT TỪ FILE PACK ZIP
# ==============================================================================
import zipfile, json, sys, sysconfig, pathlib, subprocess, torch

SP = pathlib.Path(sysconfig.get_paths()["purelib"])
# Thay FILE_ID bên dưới bằng FILE_ID của file zip bạn vừa tạo trên Google Drive
FILE_ID = "YOUR_NEW_GOOGLE_DRIVE_FILE_ID_HERE"

assert torch.cuda.get_device_capability(0) == (7, 5), f"Khong phai T4: {torch.cuda.get_device_name(0)}, doi runtime sang T4"

# 1) Tai zip
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
import gdown
gdown.download(id=FILE_ID, output="/content/pack.zip", quiet=False)

# 2) Kiem tra phien ban + giai nen truc tiep vao site-packages
with zipfile.ZipFile("/content/pack.zip") as z:
    assert z.testzip() is None, "File tai ve bi hong, tai lai"
    m = json.loads(z.read("__manifest__.json"))
    now = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda}
    bad = {k: (m[k], now[k]) for k in now if m[k] != now[k]}
    assert not bad, f"Colab da doi moi truong, .so se loi: {bad}"
    for name in z.namelist():
        if not name.startswith("__"):
            z.extract(name, SP)
print("Da giai nen vao:", SP)

# 3) Cai dependency phu
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops", "packaging", "ninja", "transformers", "triton"])

# 4) Test that tren GPU
import selective_scan_cuda, causal_conv1d_cuda
from mamba_ssm.modules.mamba_simple import causal_conv1d_fn
from mamba_ssm import Mamba
print("causal_conv1d_fn:", "OK (CUDA kernel)" if causal_conv1d_fn is not None else "None = FALLBACK, loi")
y = Mamba(d_model=64).cuda()(torch.randn(2, 128, 64, device="cuda")); y.mean().backward()
print("MAMBA OK", y.shape)
